In [ ]:
import os
import json
import pandas as pd

def jsonl_to_meta_table(
    jsonl_path: str,
    out_csv: str = "meta_v1.csv",
    out_parquet: str = "meta_v1.parquet",
    bad_images_path: str | None = None,   # optional: bad-image list (csv or txt)
):
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            nutr = obj.get("nutrition", {}) or {}

            image_path = obj.get("image_path", "")
            filename = os.path.basename(image_path)
            image_id = os.path.splitext(filename)[0] if filename else f"line_{line_no}"

            rows.append({
                "image_id": image_id,
                "image_path": image_path.replace("\\", "/"),  # normalize path separators for cross-platform use
                "dish_name": obj.get("dish", None),
                "category": obj.get("category", None),
                "cooking_method": obj.get("cooking_method", None),
                "calories_kcal": nutr.get("calories_kcal", None),
                "fat_g": nutr.get("fat_g", None),
                "protein_g": nutr.get("protein_g", None),
                "carbohydrate_g": nutr.get("carbohydrate_g", None),
                "food_prob": obj.get("food_prob", None),
                "camera_or_phone_prob": obj.get("camera_or_phone_prob", None),
                "sub_dt": obj.get("sub_dt", None),
            })

    df = pd.DataFrame(rows)

    # basic cleaning: drop rows missing key fields
    df = df.dropna(subset=["image_path", "dish_name"]).reset_index(drop=True)

    # convert sub_dt to date format (optional)
    # sub_dt looks like YYYYMMDD
    df["sub_dt"] = df["sub_dt"].astype(str)
    df.loc[df["sub_dt"].str.len() != 8, "sub_dt"] = None

    # merge bad images list (optional)
    df["bad_flag"] = False
    if bad_images_path:
        ext = os.path.splitext(bad_images_path)[1].lower()
        if ext == ".csv":
            bad_df = pd.read_csv(bad_images_path)
            # support two common column names: image_path or image_id
            if "image_path" in bad_df.columns:
                bad_set = set(bad_df["image_path"].astype(str).str.replace("\\", "/"))
                df.loc[df["image_path"].isin(bad_set), "bad_flag"] = True
            elif "image_id" in bad_df.columns:
                bad_set = set(bad_df["image_id"].astype(str))
                df.loc[df["image_id"].isin(bad_set), "bad_flag"] = True
            else:
                raise ValueError("bad_images csv must have image_path or image_id column")
        else:
            # txt: one image_path or image_id per line (auto-detected)
            with open(bad_images_path, "r", encoding="utf-8") as bf:
                bad_lines = [x.strip() for x in bf if x.strip()]
            # if entries look like paths (contain / or \) treat as path, otherwise as id
            if any(("/" in x or "\\" in x) for x in bad_lines):
                bad_set = set([x.replace("\\", "/") for x in bad_lines])
                df.loc[df["image_path"].isin(bad_set), "bad_flag"] = True
            else:
                bad_set = set(bad_lines)
                df.loc[df["image_id"].isin(bad_set), "bad_flag"] = True

    # save output
    df.to_csv(out_csv, index=False)
    df.to_parquet(out_parquet, index=False)

    print("Saved:")
    print(" -", out_csv, f"({len(df)} rows)")
    print(" -", out_parquet, f"({len(df)} rows)")
    print("Columns:", list(df.columns))

    return df


In [ ]:
df_meta = jsonl_to_meta_table(
    jsonl_path=r"C:\Users\windows\Downloads\data_100k\meta_100k.jsonl",
    out_csv=r"C:\Users\windows\Downloads\data_100k\meta_v1.csv",
    out_parquet=r"C:\Users\windows\Downloads\data_100k\meta_v1.parquet",
)


In [ ]:
import numpy as np
import pandas as pd

def sanity_check(df: pd.DataFrame, topn: int = 20):
    print("Rows:", len(df))
    print("Unique dishes:", df["dish_name"].nunique())
    print("Unique categories:", df["category"].nunique() if "category" in df else "N/A")
    print("Bad flagged:", int(df["bad_flag"].sum()) if "bad_flag" in df else "N/A")
    print()

    # missing rate
    cols = ["dish_name", "image_path", "cooking_method", "calories_kcal", "fat_g", "protein_g", "carbohydrate_g"]
    cols = [c for c in cols if c in df.columns]
    miss = df[cols].isna().mean().sort_values(ascending=False)
    print("Missing rate:")
    print((miss * 100).round(2).astype(str) + "%")
    print()

    # dish long tail (top dishes)
    vc = df["dish_name"].value_counts()
    print(f"Top {topn} dishes:")
    print(vc.head(topn))
    print()
    print("Dish count quantiles:")
    print(vc.quantile([0.5, 0.9, 0.99]).round(2))
    print()

    # nutrition range check (rough)
    nut_cols = ["calories_kcal", "fat_g", "protein_g", "carbohydrate_g"]
    nut_cols = [c for c in nut_cols if c in df.columns]
    if nut_cols:
        desc = df[nut_cols].describe(percentiles=[0.01, 0.5, 0.99]).T
        print("Nutrition summary:")
        display(desc)
        # simple outlier flag (no strong filtering, just a heads-up)
        for c in nut_cols:
            neg = int((df[c] < 0).sum()) if df[c].notna().any() else 0
            if neg:
                print(f"WARNING: {c} has {neg} negative values")
        print()

    # cooking_method normalization check
    if "cooking_method" in df.columns:
        print(f"Top {topn} cooking_method (raw):")
        print(df["cooking_method"].astype(str).str.strip().value_counts().head(topn))

sanity_check(df_meta, topn=15)


In [ ]:
import hashlib, os, pandas as pd

csv_path = "MM-Food-100K.csv"
img_root = r"C:\Users\windows\Downloads\data_100k\images"  # your image directory

df_raw = pd.read_csv(csv_path)
u = df_raw.loc[0, "image_url"]

sha1 = hashlib.sha1(u.encode("utf-8")).hexdigest()
candidate = os.path.join(img_root, sha1 + ".jpg")
print("url:", u)
print("sha1:", sha1)
print("exists:", os.path.exists(candidate), candidate)


In [ ]:
import os, json, ast, hashlib
import pandas as pd
import numpy as np

def safe_parse_list(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    if not s:
        return []
    try:
        # values look like ["a","b"], literal_eval is the most robust option
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else []
    except Exception:
        return []

def safe_parse_dict(s):
    if pd.isna(s):
        return {}
    s = str(s).strip()
    if not s:
        return {}
    # nutritional_profile looks more like a JSON string
    try:
        return json.loads(s)
    except Exception:
        # some data may use single-quote style, fall back to literal_eval
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else {}
        except Exception:
            return {}

def build_meta_v2_from_csv(csv_path, img_root, out_csv, out_parquet):
    df = pd.read_csv(csv_path)

    # unify column names (your sample uses dish_name / food_type / nutritional_profile)
    # if your actual csv uses different column names, change the mapping here
    df = df.rename(columns={
        "dish_name": "dish_name",
        "food_type": "category",
        "nutritional_profile": "nutritional_profile",
    })

    # parse fields
    df["ingredients_list"] = df["ingredients"].apply(safe_parse_list) if "ingredients" in df.columns else [[]]*len(df)
    df["portion_list"] = df["portion_size"].apply(safe_parse_list) if "portion_size" in df.columns else [[]]*len(df)
    nutr = df["nutritional_profile"].apply(safe_parse_dict) if "nutritional_profile" in df.columns else [{}]*len(df)

    df["calories_kcal"] = nutr.apply(lambda x: x.get("calories_kcal"))
    df["fat_g"] = nutr.apply(lambda x: x.get("fat_g"))
    df["protein_g"] = nutr.apply(lambda x: x.get("protein_g"))
    df["carbohydrate_g"] = nutr.apply(lambda x: x.get("carbohydrate_g"))

    # generate image_id (sha1(url)) and build local path
    df["image_id"] = df["image_url"].astype(str).apply(lambda u: hashlib.sha1(u.encode("utf-8")).hexdigest())
    df["image_path"] = df["image_id"].apply(lambda h: os.path.join(img_root, h + ".jpg")).str.replace("\\", "/")

    # keep portion as-is + generate readable text
    df["portion_text_raw"] = df["portion_list"].apply(lambda xs: "; ".join(map(str, xs)) if xs else None)
    df["ingredients_text_raw"] = df["ingredients_list"].apply(lambda xs: "; ".join(map(str, xs)) if xs else None)

    # select output columns (keep the main table flat and training-friendly)
    keep_cols = [
        "image_id", "image_url", "image_path",
        "dish_name", "category", "cooking_method", "sub_dt",
        "food_prob", "camera_or_phone_prob",
        "ingredients_list", "ingredients_text_raw",
        "portion_list", "portion_text_raw",
        "calories_kcal", "fat_g", "protein_g", "carbohydrate_g",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    meta = df[keep_cols].copy()

    # flag whether download succeeded (useful for filtering missing images)
    meta["file_exists"] = meta["image_path"].apply(lambda p: os.path.exists(p.replace("/", os.sep)))

    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    meta.to_csv(out_csv, index=False)
    meta.to_parquet(out_parquet, index=False)

    print("Saved:", out_csv, out_parquet)
    print("Rows:", len(meta))
    print("file_exists rate:", float(meta["file_exists"].mean()))
    print("portion coverage:", float(meta["portion_text_raw"].notna().mean()))
    return meta

meta_v2 = build_meta_v2_from_csv(
    csv_path="MM-Food-100K.csv",
    img_root=r"C:\Users\windows\Downloads\data_100k\images",
    out_csv=r"C:\Users\windows\Downloads\data_100k\processed\meta_v2.csv",
    out_parquet=r"C:\Users\windows\Downloads\data_100k\processed\meta_v2.parquet",
)
